# StatEO26 - Workshop Notebook: Execute, Analyse, and Visualise

This notebook is a **workshop guideline** for an end-to-end APEx scenario:

1. Discover an EO algorithm in the APEx Algorithm Catalogue.
2. Execute the service through an existing platform.
3. Publish outputs to an APEx Product Catalogue.
4. Visualise results in APEx Geospatial Explorer.

It is designed to be adapted live during the workshop.

## Pre-requisites

To get started with this notebook, please ensure that the following pre-requisites are met:

* You will need an account on CDSE to execute the openEO-based service from the APEx Algorithm Catalogue. Create your account by navigating to [this page](https://identity.dataspace.copernicus.eu/auth/realms/CDSE/account).

## Learning Goals

By the end of this session, participants should be able to:

- Navigate APEx Algorithm Catalogue metadata and identify a suitable service.
- Build and run an openEO workflow for EO-based statistics.
- Package and publish output metadata to a Product Catalogue.
- Configure a shareable map link for Geospatial Explorer-based communication.

In [1]:
# If needed, uncomment this line in a fresh environment:
# !pip install openeo pystac requests python-dotenv

## 1. Discover Candidate Algorithms (APEx Algorithm Catalogue Exploration)

This step demonstrates how to explore the existing services from the APEx Algorithm Catalogue. You can visually browse the APEx Algorithm Catalogue at https://algorithm-catalogue.apex.esa.int/. You can use the filtering methods to find the relevant service for the scenario. For example, you can view all statistics related algorithms at https://algorithm-catalogue.apex.esa.int/?labels=statistics.

/* SCREENSHOT OF CATALOGUE */

Once you have found your service, you can click on it to get the detailed information needed to execute the service. This information is available in the **Execution Information** tab on the page. 

/* SCREENSHOT OF DETAILS PAGE */

In [9]:
process_id = "worldcover_statistics"
process = "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/algorithm_catalog/vito/worldcover_statistics/openeo_udp/worldcover_statistics.json"
backend = "https://openeofed.dataspace.copernicus.eu"

## 2. Execute Workflow via openEO

Now that we have our process selected, we can start executing it through openEO on the defined backend.

In [10]:
import openeo

In [11]:
con = openeo.connect(backend).authenticate_oidc()
con

Authenticated using refresh token.


<Connection to 'https://openeofed.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [16]:
import json 

# Load AOI geometry from local GeoJSON file
with open('aoi.geojson', 'r', encoding='utf-8') as f:
    geometries = json.load(f)

eo_stat = con.datacube_from_process(
    process_id=process_id,
    namespace=process,
    # Pass AOI geometry loaded from aoi.geojson
    geometries=geometries,
)

In [22]:
# Submit batch job (adapt output format/name to your backend support)
job = eo_stat.create_job(
    format="CSV",
    title="StatEO26 - APEx Workshop EO Statistics Demo Output",
    description='APEx workshop EO-based statistics demo output'
)

job.start_and_wait()

0:00:00 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': send 'start'
0:00:18 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': created (progress 0%)
0:00:24 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': created (progress 0%)
0:00:31 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': created (progress 0%)
0:00:52 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': created (progress 0%)
0:01:03 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': created (progress 0%)
0:01:16 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': queued (progress 0%)
0:01:31 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': queued (progress 0%)
0:01:51 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': queued (progress 0%)
0:02:16 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': queued (progress 0%)
0:02:46 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': running (progress N/A)
0:03:23 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': running (progress N/A)
0:04:10 Job 'cdse-j-2604161502084bc8a861e4a26c21e820': running (progress N/A)
0:05:09 J

<BatchJob job_id='cdse-j-2604161502084bc8a861e4a26c21e820'>

In [21]:
job.get_results().download_files("./outputs")

[PosixPath('outputs/timeseries.csv'),
 PosixPath('outputs/timeseries.json'),
 PosixPath('outputs/job-results.json')]

### Post Processing

In [27]:
import csv
import json
from pathlib import Path

# Input files
csv_path = Path("outputs/timeseries.csv")
out_geojson_path = Path("outputs/timeseries.json")

features = geometries.get("features", [])

# Ensure all features have a properties object
for feature in features:
    feature.setdefault("properties", {})

# Read CSV rows
with csv_path.open("r", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

# Preferred mapping: use explicit feature index from CSV
for row in rows:
    idx = int(row["feature_index"])
    if idx < 0 or idx >= len(features):
        raise IndexError(f"feature_index {idx} is out of range for {len(features)} features.")
    for key, value in row.items():
        if key == "feature_index":
            continue
        features[idx]["properties"][key] = value

# Write merged output GeoJSON
out_geojson_path.parent.mkdir(parents=True, exist_ok=True)
with out_geojson_path.open("w", encoding="utf-8") as f:
    json.dump(geometries, f, indent=2)

print(f"Wrote: {out_geojson_path}")
print(f"Features updated: {len(features)}")

Wrote: outputs/timeseries.json
Features updated: 2


### Upload assets to APEx S3 bucket

In [32]:
import os

os.environ["APEX_S3_ACCESS_KEY_ID"] = "HPUAP2FLGTWGGPWCYEEP"
os.environ["APEX_S3_SECRET_ACCESS_KEY"] = "csgxTwHGJF3uofsgEeLh6XtGhHkia47PJreOYriK"

In [73]:
# If needed in a fresh environment:
# !pip install boto3

import os
from pathlib import Path
from urllib.parse import urlparse

import boto3
from botocore.config import Config

# Configure S3 access through environment variables
S3_CONFIG = {
    "endpoint_url": "https://obs.eu-nl.otc.t-systems.com",
    "region_name": "default",
    "bucket": "apex-prod-public",
    "prefix": "stateo26",
    "access_key": os.getenv("APEX_S3_ACCESS_KEY_ID"),
    "secret_key": os.getenv("APEX_S3_SECRET_ACCESS_KEY"),
}

# Some S3-compatible backends do not decode optional checksum/chunked framing correctly.
# Use strict settings so uploaded objects keep clean file content.
S3_CLIENT_CONFIG = Config(
    s3={
        "addressing_style": "virtual",
        "payload_signing_enabled": False,
    },
    request_checksum_calculation="when_required",
    response_checksum_validation="when_required",
)

def build_public_url(endpoint_url: str, bucket: str, key: str) -> str:
    parsed = urlparse(endpoint_url)
    return f"{parsed.scheme}://{bucket}.{parsed.netloc}/{key}"

In [74]:
# Create S3 client and upload generated assets (virtual-host style required by this endpoint)
s3_client = boto3.client(
    "s3",
    endpoint_url=S3_CONFIG["endpoint_url"],
    region_name=S3_CONFIG["region_name"],
    aws_access_key_id=S3_CONFIG["access_key"],
    aws_secret_access_key=S3_CONFIG["secret_key"],
    config=S3_CLIENT_CONFIG,
)

files_to_upload = [
    Path("outputs/timeseries.json"),
    Path("outputs/timeseries.csv"),
]

content_types = {
    ".json": "application/geo+json",
    ".csv": "text/csv",
}

upload_urls = []
for file_path in files_to_upload:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")

    # Reuse the same object key each run; uploading to an existing key overwrites it.
    object_key = f"{S3_CONFIG['prefix'].strip('/')}/{file_path.name}".strip("/")

    # Use put_object with raw bytes to avoid chunked upload framing in object content.
    with file_path.open("rb") as f:
        body = f.read()

    s3_client.put_object(
        Bucket=S3_CONFIG["bucket"],
        Key=object_key,
        Body=body,
        ContentType=content_types.get(file_path.suffix.lower(), "application/octet-stream"),
        ACL="public-read",
    )

    upload_urls.append(build_public_url(S3_CONFIG["endpoint_url"], S3_CONFIG["bucket"], object_key))
    print(f"Uploaded (public-read, overwrite enabled): {file_path}")
    print(f"Public URL: {upload_urls[-1]}")

Uploaded (public-read, overwrite enabled): outputs/timeseries.json
Public URL: https://apex-prod-public.obs.eu-nl.otc.t-systems.com/stateo26/timeseries.json
Uploaded (public-read, overwrite enabled): outputs/timeseries.csv
Public URL: https://apex-prod-public.obs.eu-nl.otc.t-systems.com/stateo26/timeseries.csv


## 3. Publish to APEx Product Catalogue

Next we can use the outputs from the openEO job to directly publish our results to an APEx Product Catalogue. 

In [75]:
stac_collection_id = f"apex-workshop-eo-stats-{job.job_id}"

In [76]:
import pystac

# Read the stac metadata from the outpus/job-results.json (STAC-like structure from openEO results endpoint)
with open("outputs/job-results.json", "r", encoding="utf-8") as f:
    stac_metadata_dict = json.load(f)
    
stac_metadata_dict['id'] = stac_collection_id

# Replace the paths in the STAC metadata with the S3 URLs of the uploaded assets
for asset_key, asset in stac_metadata_dict.get("assets", {}).items():
    # Assuming the asset href is the filename (e.g., "timeseries.json"), find the corresponding uploaded key
    matching_url = [key for key in upload_urls if key.endswith(asset_key)]
    if matching_url:
        asset["href"] = matching_url[0]
    else:
        print(f"Warning: No matching uploaded key found for asset href: {asset['href']}")

collection = pystac.Collection.from_dict(stac_metadata_dict)
collection.title = f'StatEO26 - APEx Workshop EO Statistics Output - {job.job_id}'
collection.license = 'CC-BY-4.0'

print('Collection ID:', collection.id)
print('Number of links:', len(collection.links))

Collection ID: apex-workshop-eo-stats-cdse-j-2604161502084bc8a861e4a26c21e820
Number of links: 7


First we need to obtain an access token for the APEx Catalogue API. This will be done by exchanging the client credentials, shared by the APEx team, with an actual token.

**Option A: set credentials in your terminal before starting Jupyter**

> These commands are intended for the terminal, not for a Python notebook cell.

```bash
export APEX_CLIENT_ID="your-client-id"
export APEX_CLIENT_SECRET="your-client-secret"
```

**Option B: set credentials for the current notebook session**

> Run the next cell and replace the placeholder values with your actual credentials.


In [77]:
os.environ["APEX_CLIENT_ID"] = "stateo26-catalogue-prod-m2m"
os.environ["APEX_CLIENT_SECRET"] = "rBHxrDwL1t4gqiginFJMQVz2Ua3del8b"

In [78]:
import os
import requests

apex_client_id = os.getenv('APEX_CLIENT_ID')
apex_client_secret = os.getenv('APEX_CLIENT_SECRET')

if not apex_client_id or not apex_client_secret:
    raise ValueError(
        'Set APEX_CLIENT_ID and APEX_CLIENT_SECRET first, either in your terminal or in the notebook setup cell above.'
    )

realm_url = 'https://auth.apex.esa.int/realms/apex'
token_url = f"{realm_url}/protocol/openid-connect/token"

token_resp = requests.post(
    token_url,
    data={
        'grant_type': 'client_credentials',
        'client_id': apex_client_id,
        'client_secret': apex_client_secret,
    },
    headers={'Content-Type': 'application/x-www-form-urlencoded'},
    timeout=30,
)
token_resp.raise_for_status()

token_payload = token_resp.json()
token = token_payload['access_token']

print('Token endpoint:', token_url)
print('Token acquired:', bool(token))

Token endpoint: https://auth.apex.esa.int/realms/apex/protocol/openid-connect/token
Token acquired: True


In [79]:
# Use the exchanged token above, or fall back to APEX_CATALOG_TOKEN from the environment
token = globals().get('token')

# Setting the Authorization header for the catalogue API
headers = {'Content-Type': 'application/json'}
if token:
    headers['Authorization'] = f'Bearer {token}'


In [80]:
# Setting the permissions for the collection, allowing anyone to read but only the current user to write
payload = collection.to_dict()
default_auth = {
    "_auth": {
        "read": ["anonymous"],
        "write": [os.getenv('APEX_CLIENT_ID')]
    }
}
payload.update(default_auth)


In [81]:

import json

# Publish the collection to the APEx catalogue (adjust URL if needed)
publish_resp = requests.post(
    'https://catalogue.stateo26.apex.esa.int/collections',
    headers=headers,
    data=json.dumps(payload),
    timeout=60
)
print('Publish status:', publish_resp.status_code)
print('Publish response:', publish_resp.text[:1000])


Publish status: 201
Publish response: {"id":"apex-workshop-eo-stats-cdse-j-2604161502084bc8a861e4a26c21e820","description":"APEx workshop EO-based statistics demo output","stac_version":"1.1.0","links":[{"rel":"items","type":"application/geo+json","href":"https://catalogue.stateo26.apex.esa.int/collections/apex-workshop-eo-stats-cdse-j-2604161502084bc8a861e4a26c21e820/items"},{"rel":"parent","type":"application/json","href":"https://catalogue.stateo26.apex.esa.int/"},{"rel":"root","type":"application/json","href":"https://catalogue.stateo26.apex.esa.int/"},{"rel":"self","type":"application/json","href":"https://catalogue.stateo26.apex.esa.int/collections"},{"href":"https://catalogue.stateo26.apex.esa.int/v200_N39E012","rel":"derived_from","type":"application/json","title":"Derived from v200_N39E012"},{"href":"https://openeo.dataspace.copernicus.eu/openeo/1.1/jobs/j-2604161326524a72a26f7e726a0153f9/results/NjM5MTg1MWYtOTA0Mi00MTA4LThiMmEtM2RkMmU4YTlkZDBi/a4b139ea4394361488eb03115639a37a

The new collection is now available in the APEx Product Catalogue at https://browser.stateo26.apex.esa.int/

In [ ]:
# # Clean up by deleting the collection again (optional)
# delete_resp = requests.delete(
#     f"https://catalogue.stateo26.apex.esa.int/collections/{collection.id}",
#     headers=headers,
#     timeout=30
# )
# print('Delete status:', delete_resp.status_code)

Delete status: 204


## 5. Visualise in APEx Geospatial Explorer

Use the published collection/item URL or direct asset URL to build a shareable workshop link.
Exact query parameters may differ per Explorer deployment/version.

In [ ]:
# Example: link to records endpoint filtered by collection id
collection_query_url = f"{CFG['product_catalogue_records_url']}?q={quote(CFG['stac_collection_id'])}"

# Generic explorer handoff URL pattern (adapt to actual explorer route/params)
explorer_link = f"{CFG['geospatial_explorer_url']}?catalog={quote(collection_query_url, safe=':/?=&')}"

print('Collection query URL:')
print(collection_query_url)
print('\nProposed Explorer link:')
print(explorer_link)